# Part 3 — Option A: a verifier, and the evidence it needs

**What changed** (branch `feat/verified-sections`, see `improvement/0002-verified-sections.patch`):
1. *Evidence.* Forecast columns are labelled from the base year (FY+1..FY+3, e.g. `2027E–2029E` for NVIDIA) instead
   of a hard-coded `2025E–2027E`, and every writer prompt starts with reading notes: the report date, which columns
   are reported and which are the pipeline's own forecast (with its assumptions), the CAGR period, which peer year is
   projected, and that the data contains no guidance, consensus or news.
2. *Verifier* (`modules/claim_verifier.py`, `--enable-verifier`, default `gpt-4.1-mini`). After each section is
   written, it checks the section against the writer's data (plus the report date and notes) and replaces sentences
   that mislabel reported vs. forecast figures, misattribute the CAGR period, misquote numbers or comparisons, or cite
   sources that do not exist. Corrections are exact sentence replacements, logged in `verifier_log.json`. Statements
   the data does not cover are listed, not deleted.

**Experiment** (`audit/part3_experiment.py`): 2 x 2 on the same saved data for 10 companies x 7 sections.
`C0` original prompt / `C1` annotated prompt, each with and without the verifier. The writer is called exactly as the
pipeline calls it (gpt-4.1, temperature 0.7). Each text is scored against the prompt its writer saw by the Part 2.3
scorer (gpt-4.1 judge + rules), and by the LLM-free status-error counter that neither the writer nor the verifier
uses. The five healthcare names are a held-out set: they were never used while designing or tuning
the changes (the pilots used NVDA and AMD).

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "audit" else Path.cwd()
sys.path.insert(0, str(ROOT / "audit"))
import grounding as g

pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 140)
S = pd.read_csv(ROOT / "audit" / "part3_sections.csv")
C = pd.read_csv(ROOT / "audit" / "part3_claims.csv").fillna("")
T = [json.loads(l) for l in open(ROOT / "audit" / "part3_texts.jsonl", encoding="utf-8")]
ORDER = ["C0", "C0+V", "C1", "C1+V"]
S["set"] = np.where(S["held_out"], "held-out (healthcare)", "development")


def relabel(fm):  # as in part3_experiment.py: C1 tables carry FY+1..FY+3 labels
    base = max(int(c[:4]) for c in fm.columns if c.endswith("A") and c[:4].isdigit())
    return fm.rename(columns={"2025E": f"{base + 1}E", "2026E": f"{base + 2}E", "2027E": f"{base + 3}E"})


# Recompute the LLM-free counter from the texts so the notebook always uses the current counter.
TABLES = {}
for t in S["ticker"].unique():
    t0 = g.load_tables(g.OUT / t)
    TABLES[(t, "C0")], TABLES[(t, "C1")] = t0, {**t0, "financial_metrics": relabel(t0["financial_metrics"])}
TEXT = {(r["ticker"], r["section"], r["condition"]): r["text"] for r in T}
counts = [g.status_errors(TEXT[(r.ticker, r.section, r.condition)], TABLES[(r.ticker, r.condition[:2])])
          for r in S.itertuples()]
for k in ("actual_as_forecast", "cagr_period", "invented_source", "total"):
    S[f"counter_{k}"] = [c[k] for c in counts]
print(f"{S[['ticker', 'section']].drop_duplicates().shape[0]} company-sections x {S['condition'].nunique()} conditions; "
      f"{len(C)} claims scored; verifier errors: {S['verifier_error'].notna().sum()}")

70 company-sections x 4 conditions; 3097 claims scored; verifier errors: 1


## 1. Is the counter itself trustworthy?

Before using the counter as an outcome, it has to pass the same test as any checker: on sentences generated
mechanically from each company's own table it must report nothing (no false positives), and on the same sentences
with one known error injected it must report every one.

In [2]:
def oracle_and_faults(tables):
    fm = tables["financial_metrics"].set_index("metrics")
    a_cols = [c for c in fm.columns if c.endswith("A") and c[:4].isdigit()]
    e_cols = [c for c in fm.columns if c.endswith("E") and c[:4].isdigit()]
    a, last, first = a_cols[-1], int(a_cols[-1][:4]), a_cols[0]
    rev_a = float(fm.loc["Revenue", a]) / 1e9
    rev_e = float(fm.loc["Revenue", e_cols[0]]) / 1e9
    marg = str(fm.loc["EBITDA Margin", a])
    e_marg = {str(fm.loc["EBITDA Margin", c]) for c in e_cols}
    cagr = str(fm["CAGR"].iloc[0])
    oracle = [f"Revenue was ${rev_a:.1f} billion in FY{last}.",
              f"EBITDA margin was {marg} in FY{last}.",
              f"The pipeline's forecast for {e_cols[0]} revenue is ${rev_e:.1f} billion.",
              f"Revenue CAGR was {cagr} from {first[:4]} to {last}."]
    faults = {"actual presented as forecast": f"Revenue is projected to reach ${rev_a:.1f} billion in {last}.",
              "E label on a reported value": f"Revenue of ${rev_a:.1f} billion ({last}E) shows strong momentum.",
              "CAGR attributed to forecast years": f"Revenue is set to grow at a {cagr} CAGR through {last + 3}.",
              "invented consensus": f"Consensus expects revenue of ${rev_e:.1f} billion next year."}
    if marg not in e_marg:  # skip when the reported margin also appears as a forecast value
        faults["reported margin presented as estimate"] = f"EBITDA margin is expected to reach {marg} in {last}."
    return oracle, faults


rows = []
for t in S["ticker"].unique():
    tables = g.load_tables(g.OUT / t)
    oracle, faults = oracle_and_faults(tables)
    fp = sum(g.status_errors(s, tables)["total"] for s in oracle)
    hits = {k: g.status_errors(v, tables)["total"] > 0 for k, v in faults.items()}
    rows.append({"ticker": t, "false positives (oracle)": fp, "faults injected": len(hits),
                 "faults detected": sum(hits.values()), "missed": ", ".join(k for k, v in hits.items() if not v) or "-"})
selfcheck = pd.DataFrame(rows).set_index("ticker")
print("TOTAL false positives:", selfcheck["false positives (oracle)"].sum(),  # see Findings for the precision on real text
      "| detected:", selfcheck["faults detected"].sum(), "/", selfcheck["faults injected"].sum())
selfcheck

TOTAL false positives: 0 | detected: 50 / 50


,false positives (oracle),faults injected,faults detected,missed
ticker,,,,
AAPL,0,5,5,-
ABBV,0,5,5,-
AMD,0,5,5,-
GOOGL,0,5,5,-
INTC,0,5,5,-
JNJ,0,5,5,-
MRNA,0,5,5,-
NVDA,0,5,5,-
PFE,0,5,5,-


## 2. Results by condition

Rates are pooled over claims; the 95% interval bootstraps over the 70 company-sections. *Supported claims* is an
absolute count: a fix that deleted content could lower the error rate while making the report emptier.

In [3]:
def boot(df, stat, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    units = df.groupby(["ticker", "section"])
    keys = list(units.groups)
    vals = []
    for _ in range(n):
        pick = [keys[i] for i in rng.integers(0, len(keys), len(keys))]
        vals.append(stat(pd.concat([units.get_group(k) for k in pick])))
    return np.percentile(vals, [2.5, 97.5])


def summarize(df):
    out = {}
    for cond in ORDER:
        d = df[df.condition == cond]
        rate = lambda x, col: x[col].sum() / x["claims"].sum()
        lo, hi = boot(d, lambda x: rate(x, "contradicted"))
        out[cond] = {
            "claims": d["claims"].sum(), "supported claims": d["supported"].sum(),
            "grounding rate": rate(d, "supported"), "unsupported": rate(d, "unsupported"),
            "contradicted": rate(d, "contradicted"), "contradicted 95% CI": f"{lo:.1%}-{hi:.1%}",
            "quant. claims contradicted": d["quant_contradicted"].sum() / max(d["quant_claims"].sum(), 1),
            "status errors per section (counter)": d["counter_total"].mean(),
            "sections with >=1 status error": (d["counter_total"] > 0).mean(),
            "words per section": d["words"].mean(),
        }
    return pd.DataFrame(out)


summarize(S).round(3)

,C0,C0+V,C1,C1+V
claims,717,740,824,816
supported claims,413,506,680,686
grounding rate,0.576011,0.683784,0.825243,0.840686
unsupported,0.260809,0.259459,0.15534,0.145833
contradicted,0.16318,0.056757,0.019417,0.01348
contradicted 95% CI,13.2%-19.7%,4.0%-7.6%,1.2%-2.7%,0.6%-2.1%
quant. claims contradicted,0.235174,0.076132,0.022617,0.015974
status errors per section (counter),1.471429,0.271429,0.028571,0.028571
sections with >=1 status error,0.771429,0.228571,0.028571,0.028571
words per section,226.842857,224.014286,243.714286,240.314286


In [4]:
for name, d in S.groupby("set"):
    print(f"\n=== {name}")
    print(summarize(d).round(3).to_string())


=== development


                                              C0        C0+V          C1       C1+V
claims                                       373         376         430        425
supported claims                             210         262         352        355
grounding rate                          0.563003    0.696809    0.818605   0.835294
unsupported                             0.276139    0.255319     0.15814   0.150588
contradicted                            0.160858    0.047872    0.023256   0.014118
contradicted 95% CI                  11.8%-20.7%   2.7%-7.2%   1.2%-3.5%  0.5%-2.6%
quant. claims contradicted              0.238683    0.058577    0.028662   0.018519
status errors per section (counter)     1.742857    0.257143         0.0        0.0
sections with >=1 status error               0.8    0.228571         0.0        0.0
words per section                     230.628571  225.285714  240.285714      236.4

=== held-out (healthcare)


                                              C0        C0+V          C1        C1+V
claims                                       344         364         394         391
supported claims                             203         244         328         331
grounding rate                          0.590116     0.67033    0.832487    0.846547
unsupported                             0.244186    0.263736    0.152284    0.140665
contradicted                            0.165698    0.065934    0.015228    0.012788
contradicted 95% CI                  12.1%-21.5%   4.0%-9.7%   0.5%-2.7%   0.5%-2.4%
quant. claims contradicted              0.231707    0.093117    0.016393    0.013245
status errors per section (counter)          1.2    0.285714    0.057143    0.057143
sections with >=1 status error          0.742857    0.228571    0.057143    0.057143
words per section                     223.057143  222.742857  247.142857  244.228571


## 3. Paired comparison against C0

Same company-section, same data: the mean change per section in contradicted claims, counter-detected status errors
and supported claims, with bootstrap 95% intervals.

In [5]:
def paired(df, cond, col):
    a = df[df.condition == "C0"].set_index(["ticker", "section"])[col]
    b = df[df.condition == cond].set_index(["ticker", "section"])[col]
    diff = (b - a).dropna()
    rng = np.random.default_rng(1)
    bs = [diff.sample(len(diff), replace=True, random_state=int(rng.integers(1e9))).mean() for _ in range(2000)]
    return f"{diff.mean():+.2f} ({np.percentile(bs, 2.5):+.2f} to {np.percentile(bs, 97.5):+.2f})"


rows = []
for cond in ORDER[1:]:
    for name, d in [("all", S)] + list(S.groupby("set")):
        rows.append({"condition": cond, "set": name,
                     "contradicted claims / section": paired(d, cond, "contradicted"),
                     "status errors / section (counter)": paired(d, cond, "counter_total"),
                     "supported claims / section": paired(d, cond, "supported")})
pd.DataFrame(rows).set_index(["condition", "set"])

contradicted claims / section status errors / section (counter) supported claims / section
condition set                                                                                                             
C0+V      all                          -1.07 (-1.40 to -0.76)            -1.20 (-1.60 to -0.89)     +1.33 (+0.87 to +1.83)
          development                  -1.20 (-1.69 to -0.74)            -1.49 (-2.09 to -0.97)     +1.49 (+0.71 to +2.29)
          held-out (healthcare)        -0.94 (-1.37 to -0.57)            -0.91 (-1.31 to -0.60)     +1.17 (+0.66 to +1.77)
C1        all                          -1.44 (-1.81 to -1.10)            -1.44 (-1.84 to -1.11)     +3.81 (+3.03 to +4.66)
          development                  -1.43 (-1.94 to -0.94)            -1.74 (-2.31 to -1.20)     +4.06 (+2.94 to +5.23)
          held-out (healthcare)        -1.46 (-1.94 to -1.06)            -1.14 (-1.54 to -0.80)     +3.57 (+2.49 to +4.60)
C1+V      all                          -1.51 (-1.87 to -1.17)            -1.44 (-1.84 to -1.11)     +3.90 (+3.13 to +4.76)
          development                  -1.54 (-2.06 to -1.06)            -1.74 (-2.31 to -1.20)     +4.14 (+2.94 to +5.40)
          held-out (healthcare)        -1.49 (-1.97 to -1.09)            -1.14 (-1.54 to -0.80)     +3.66 (+2.60 to +4.71)

In [6]:
by_sec = S.groupby(["section", "condition"]).apply(
    lambda d: d["contradicted"].sum() / d["claims"].sum(), include_groups=False).unstack()[ORDER]
by_sec.loc["ALL"] = [S[S.condition == c]["contradicted"].sum() / S[S.condition == c]["claims"].sum() for c in ORDER]
by_sec.round(3)

condition,C0,C0+V,C1,C1+V
section,,,,
company_overview,0.086,0.035,0.019,0.013
competitor_analysis,0.120,0.056,0.056,0.035
investment_overview,0.260,0.067,0.000,0.007
major_takeaways,0.141,0.079,0.000,0.000
risks,0.164,0.042,0.012,0.012
tagline,0.151,0.075,0.000,0.000
valuation_overview,0.238,0.064,0.031,0.019
ALL,0.163,0.057,0.019,0.013


## 4. What the verifier did

In [7]:
logs = [(r["ticker"], r["section"], r["condition"], r["verifier_log"]) for r in T if r["verifier_log"]]
corr = pd.DataFrame([{"ticker": t, "section": s, "condition": c, **x} for t, s, c, l in logs for x in l["corrections"]])
print("sections checked:", len(logs), "| corrections applied:", len(corr),
      "| proposed but not applied (sentence not found verbatim):", sum(len(l["not_applied"]) for *_, l in logs),
      "| statements listed as unverified:", sum(len(l["unverified"]) for *_, l in logs))
print(pd.crosstab(corr["problem"], corr["condition"], margins=True))

sections checked: 140 | corrections applied: 399 | proposed but not applied (sentence not found verbatim): 313 | statements listed as unverified: 628
condition            C0+V  C1+V  All
problem                             
cagr_period             8     1    9
invented_source        51    37   88
period                 15     4   19
quantifier              0     1    1
status                150    37  187
value_or_comparison    42    53   95
All                   266   133  399


In [8]:
pd.set_option("display.max_colwidth", 200)
corr.sample(min(12, len(corr)), random_state=3)[["ticker", "section", "condition", "problem", "original", "corrected"]]

,ticker,section,condition,problem,original,corrected
359,UNH,competitor_analysis,C0+V,value_or_comparison,"Valuation metrics show UNH trades at a premium to peers: its EV/EBITDA ratios (14.9x–18.6x from 2023–2025) are consistently higher than PFE and in line with, or slightly above, JNJ.","Valuation metrics show UNH trades at a premium to some peers: its EV/EBITDA ratios (14.9x–18.6x from 2023–2025) are higher than PFE in 2024 and 2025 and in line with, or slightly above, JNJ."
16,AAPL,investment_overview,C0+V,invented_source,"Consensus estimates project a revenue CAGR of 3.3% through 2027, with contribution and EBITDA margins reaching nearly 50% and 44.8%, respectively.","The reported revenue CAGR was 3.3% from 2021 to 2025, with contribution and EBITDA margins forecast to reach nearly 50% and 44.8%, respectively, by 2027."
364,UNH,investment_overview,C0+V,status,"Revenue increased from $288 billion in 2021 to $372 billion in 2023 and is forecast to reach $518 billion by 2027, reflecting the company’s strong market position and continued expansion across he...","Revenue increased from $288 billion in 2021 to $372 billion in 2023 and is this pipeline's forecast to reach $518 billion by 2027, reflecting the company’s strong market position and continued exp..."
82,AMD,company_overview,C0+V,status,"EPS also fell, but is projected to recover strongly as new product launches and AI-driven demand accelerate growth, with EPS expected to reach $3.18 by 2027E.","EPS also fell, but is projected by this pipeline to recover strongly, with EPS reaching $3.18 by 2027."
107,AMD,major_takeaways,C0+V,status,"EBITDA margin contracted from 25.3% in 2021 to 18.3% in 2023 but is forecasted to recover sharply, reaching 42.0% by 2027. This expansion highlights AMD’s improving profitability outlook and posit...","EBITDA margin contracted from a reported 25.3% in 2021 to a reported 18.3% in 2023 but this pipeline's forecast shows it recovering sharply, reaching 42.0% by 2027. This expansion highlights AMD’s..."
217,JNJ,investment_overview,C0+V,status,"Revenue grew from $79.99 billion in 2023 to $88.82 billion in 2024, an 11% increase, with management guiding for a 6.2% CAGR through 2027.","Revenue grew from $79.99 billion reported in 2023 to $88.82 billion reported in 2024, an 11% increase, with a reported CAGR of 6.2% from 2022 to 2025."
395,UNH,valuation_overview,C1+V,status,"EBITDA margin declined sharply from 9.8% in 2023A to 5.2% in 2025A, with a slow recovery expected (8.2% by 2028E).","EBITDA margin declined sharply from 9.8% in 2023A to 5.2% in 2025A, with a slow recovery expected (8.2% this pipeline's forecast for 2028E)."
56,ABBV,investment_overview,C0+V,invented_source,"However, the company is expected to rebound, with consensus estimates indicating a return to growth (3.7% in 2024 and 8.6% in 2025), and a revenue CAGR of 2.1% through 2027.","However, the company is expected to rebound, with reported revenue growth of 3.7% in 2024 and 8.6% in 2025, and a revenue CAGR of 2.1% from 2021 to 2025."
250,MRNA,competitor_analysis,C0+V,invented_source,"Meanwhile, JNJ and PFE maintain robust profitability, with JNJ’s EBITDA expected to exceed $40B by 2025 and margins remaining strong.","Meanwhile, JNJ maintains robust profitability, with EBITDA exceeding $40B by 2025."
40,ABBV,competitor_analysis,C0+V,cagr_period,"Analyzing recent performance and projections, AbbVie’s revenue has remained relatively stable, with a modest 2.1% CAGR from 2021 to 2027.","Analyzing recent performance and projections, AbbVie’s revenue has remained relatively stable, with a modest 2.1% CAGR from 2021 to 2025."


## Findings

**Instrument check.** The LLM-free counter reports 0 false positives on table-generated sentences and detects 50/50
injected status errors across the ten companies. On real text its precision is lower: a manual check of 10 flagged C0
sentences found 11 of 13 counted errors to be real (the two misses were a correctly derived forecast-period CAGR and a
number that coincided with a reported value). It is a corroborating measure; the judge is the primary one.

1. **Labelling the evidence does most of the work.** Annotating the prompt (C1) cuts contradicted claims from 16.3% to
   1.9% and counter-detected status errors from 1.47 to 0.03 per section (sections with any such error: 77% to 3%).
   The grounding rate rises from 58% to 83%, and unsupported claims fall from 26% to 16%: told what the data is, the
   writer uses more of it. This is not bought by writing less. Supported claims rise from 413 to 680 and sections get
   slightly longer (227 to 244 words). **The effect is the same on the held-out healthcare names** (16.6% to 1.5%
   contradicted), which were never used while designing or tuning the changes.
2. **A verifier on its own helps, but only with the context the writer lacked.** On the original prompt (C0+V) it cuts
   contradicted claims to 5.7%. In the pilot, without the report date, it repeated the writer's mistake: it rewrote
   "consensus estimates project revenues to climb to $130.5 billion in 2025" as "revenues are projected to climb
   to $130.5 billion in 2025", still presenting a reported figure as a forecast. With the date it fixes this class of
   error.
3. **On top of the annotation, the verifier adds little.** Contradicted claims go from 1.9% to 1.3%, but the intervals
   overlap (1.2–2.7% vs 0.6–2.1%) and the counter is unchanged at 0.03 errors per section. Its cost is one gpt-4.1-mini
   call per section. It is worth keeping as a cheap safety net and for writers whose prompt cannot be changed; it is
   not where the improvement comes from.
4. **How the verifier behaves.** It applied 399 corrections, mostly status (187), invented sources (88) and
   values/comparisons (95). Another 313 proposed corrections were not applied because the model did not quote the
   original sentence verbatim. That is safe by design, but it wastes 44% of its output. Some "corrections" are noise,
   such as replacing $26.9 billion with $26.914 billion. It listed 628 statements as unverified rather than deleting
   them.

**What it still gets wrong**
- Unsupported background claims remain (15% in C1+V). The company overview is still asked for products and strategy
  that the data does not contain. Fixing that needs either evidence for it (segments, filings text) or a narrower
  prompt.
- Step 2's `--enable-text-regeneration` rewrites two sections with gpt-4o-mini outside both fixes. It should be turned
  off, or routed through the same prompt and verifier.
- Judge and writer are the same model family; the judge was 91% accurate on the Part 2.3 review sample and lenient
  rather than strict. Each condition has one sample per company-section; the stability runs show section-level
  results vary a lot between samples, which is why the comparison pools 70 of them.
- Out of scope and unchanged: forecast EBITDA still omits R&D, and the valuation engine is still a fixed 10.84x
  multiple (Part 2.2).
